[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_01/MFM_ch1_inference/MFM_ch1_inference.ipynb)

# MFM_ch1_inference

Inference for stylised facts under heavy tails and dependence: quantile-based skewness and kurtosis (Hinkley, Crow-Siddiqui) with moving-block bootstrap intervals, heteroskedasticity-robust lag-1 autocorrelation, Hill tail indices with i.i.d. and block-bootstrap standard errors and a common-threshold comparison of the two tails, and the long-memory parameter d of absolute returns (local Whittle, log-periodogram) before and after a CUSUM-of-squares variance break.

*Modelling Financial Markets (MFM), Chapter 1: Stylised Facts and Returns. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_1'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
Teal     = '#17A2B8'

ASSETS = ['sp500', 'bettr', 'btc', 'eurron', 'gold']
COLORS = {'sp500': MainBlue, 'bettr': IDAred, 'btc': Amber, 'eurron': Forest, 'gold': Purple}
PERIODS = {'sp500': 252, 'bettr': 252, 'btc': 365, 'eurron': 252, 'gold': 252}


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# copia locala a folderului data/market, daca notebook-ul ruleaza din repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')

SOURCES = {
    'sp500':  ('GSPC.INDX',    'close', '1990-01-01'),
    'bettr':  ('BETTR.INDX',   'close', '2014-09-01'),
    'btc':    ('BTC-USD.CC',   'close', '2014-09-17'),
    'eurron': ('REF:EUR',      'close', '2005-07-01'),   # de la introducerea leului nou (RON), 1 iulie 2005
    'gold':   ('XAUUSD.FOREX', 'close', '1990-01-01'),
}

LABELS = {'sp500': 'S&P 500', 'bettr': 'BET-TR (Romania)', 'btc': 'Bitcoin',
          'eurron': 'EUR/RON', 'gold': 'Gold'}

# cotatii OTC (aur): sesiunile partiale de sambata/duminica rup randamentul vineri -> luni,
# deci pastram doar zilele lucratoare (aceeasi conventie ca in Capitolul 0)
WEEKDAYS_ONLY = {'gold'}

_CACHE = {}


def read_market(symbol):
    """Citeste data/market/<SIMBOL>.csv local sau din repo-ul GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    df = pd.read_csv(src, parse_dates=['date']).set_index('date').sort_index()
    df.index.name = 'Date'
    return df


def read_reference_rate(currency='EUR', start='2005-07-01', end='2026-09-18'):
    """Cursul oficial de referinta RON (arhive XML anuale BNR)."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    import re
    import urllib.request
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    df = pd.DataFrame(rows, columns=['Date', 'Close']).drop_duplicates('Date').set_index('Date')
    df.index = pd.to_datetime(df.index)
    _CACHE[key] = df.sort_index().loc[start:end]
    return _CACHE[key]


def load_data(name='sp500'):
    """OHLCV zilnic cu coloane Open/High/Low/Close/Volume (Close = pretul folosit)."""
    symbol, col, start = SOURCES[name]
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start)
    d = read_market(symbol).loc[start:]
    if name in WEEKDAYS_ONLY:
        d = d[d.index.dayofweek < 5]
    out = pd.DataFrame({'Open': d['open'], 'High': d['high'], 'Low': d['low'],
                        'Close': d[col], 'Volume': d['volume']})
    return out[out['Close'] > 0].dropna(subset=['Close'])


def load_close(name):
    """Pretul de inchidere, folosit in toate graficele."""
    return load_data(name)['Close'].dropna()


closes = {a: load_close(a) for a in ASSETS}
rets = {a: np.log(c).diff().dropna() for a, c in closes.items()}
# gold (weekdays only) is annualised with its actual number of observations per year
PERIODS['gold'] = len(rets['gold']) / ((closes['gold'].index[-1] - closes['gold'].index[0]).days / 365.25)
spx_ohlc = load_data('sp500')
for a in ASSETS:
    print(f'{LABELS[a]:18s}', rets[a].index[0].date(), '->', rets[a].index[-1].date(), len(rets[a]), 'obs')

## Helper functions

In [ ]:
def hill_estimator(x, k):
    """alpha_hat(k) = [ (1/k) sum_{i=1..k} ln(X_(i) / X_(k+1)) ]^{-1}, X_(1) >= X_(2) >= ..."""
    x = np.sort(np.asarray(x)[np.asarray(x) > 0])[::-1]
    return 1.0 / np.mean(np.log(x[:k] / x[k]))


def moving_block_indices(T, block, rng):
    """Indicii unui esantion bootstrap pe blocuri mobile (moving-block) de lungime `block`."""
    nb = int(np.ceil(T / block))
    st = rng.integers(0, T - block + 1, nb)      # toate cele T - block + 1 inceputuri valide
    return (st[:, None] + np.arange(block)[None, :]).ravel()[:T]


def periodogram(x):
    x = np.asarray(x) - np.mean(x)
    T = len(x)
    return 2 * np.pi * np.arange(T) / T, np.abs(np.fft.fft(x)) ** 2 / (2 * np.pi * T)


def local_whittle(x, m):
    """Robinson (1995): d minimizeaza R(d) = ln(mean(lambda_j^{2d} I_j)) - 2d mean(ln lambda_j); EE = 1/(2 sqrt m)."""
    from scipy.optimize import minimize_scalar
    lam, I = periodogram(x)
    l, Ij = lam[1:m + 1], I[1:m + 1]
    R = lambda d: np.log(np.mean(l ** (2 * d) * Ij)) - 2 * d * np.mean(np.log(l))
    return minimize_scalar(R, bounds=(-0.49, 0.99), method='bounded').x, 1 / (2 * np.sqrt(m))


def gph(x, m):
    """Geweke & Porter-Hudak (1983): panta lui ln I_j pe -2 ln(2 sin(lambda_j/2)); EE = pi/sqrt(24 m)."""
    lam, I = periodogram(x)
    X = -2 * np.log(2 * np.sin(lam[1:m + 1] / 2))
    return np.polyfit(X, np.log(I[1:m + 1]), 1)[0], np.pi / np.sqrt(24 * m)


def cusum_squares_break(r):
    """Data rupturii in varianta necoditionata: argmax |C_k/C_T - k/T| (Inclan & Tiao 1994)."""
    x = np.asarray(r) - np.mean(r)
    C = np.cumsum(x ** 2)
    T = len(x)
    D = C / C[-1] - np.arange(1, T + 1) / T
    j = int(np.argmax(np.abs(D)))
    return j, np.sqrt(T / 2) * np.abs(D[j])

## Analysis

In [ ]:
def hill_threshold(x, u):
    """Hill cu prag fix u: k = #{x > u}, alpha_hat = k / sum ln(x_i / u) pe x_i > u."""
    x = np.asarray(x)
    e = x[x > u]
    return len(e) / np.sum(np.log(e / u))


def hill_inference(frac=0.025, n_boot=500, block=20, seed=42):
    """Hill pentru |r_t| la k = frac*n: IC i.i.d. alpha(1 +/- 1.96/sqrt(k)), EE bootstrap pe blocuri
    si comparatia cu cozile unilaterale la ACELASI prag u (min al pragurilor unilaterale)."""
    rng = np.random.default_rng(seed)
    rows = []
    for a in ASSETS:
        r = rets[a].values
        n = len(r)
        k = int(frac * n)
        al = hill_estimator(np.abs(r), k)
        boot = [hill_estimator(np.abs(r)[moving_block_indices(n, block, rng)], k) for _ in range(n_boot)]
        u = min(np.sort(-r)[::-1][k], np.sort(r)[::-1][k])
        kL, kG, kA = int((-r > u).sum()), int((r > u).sum()), int((np.abs(r) > u).sum())
        rows.append({'asset': LABELS[a], 'n': n, 'k': k, 'alpha_abs': al, 'se_iid': al / np.sqrt(k),
                     'se_block': np.std(boot, ddof=1), 'ci_lo': al * (1 - 1.96 / np.sqrt(k)),
                     'ci_hi': al * (1 + 1.96 / np.sqrt(k)), 'alpha_loss_k': hill_estimator(-r, k),
                     'alpha_gain_k': hill_estimator(r, k), 'u_common_pct': 100 * u, 'k_abs_u': kA,
                     'alpha_abs_u': hill_threshold(np.abs(r), u), 'alpha_loss_u': hill_threshold(-r, u),
                     'alpha_gain_u': hill_threshold(r, u)})
    t = pd.DataFrame(rows).set_index('asset')
    t.to_csv(os.path.join(TABLE_DIR, 'ch1_hill_inference.csv'), float_format='%.4g')
    return t


def robust_moments(n_boot=1000, block=20, seed=42):
    """Asimetrie si aplatizare pe cuantile (Kim & White 2004): Hinkley (5%-95%) si Crow-Siddiqui
    (2.5%, 97.5% fata de quartile, minus 2.91, valoarea distributiei Normale); IC bootstrap pe blocuri."""
    def qstats(v):
        q = np.quantile(v, [0.025, 0.05, 0.25, 0.5, 0.75, 0.95, 0.975])
        return ((q[5] + q[1] - 2 * q[3]) / (q[5] - q[1]), (q[6] - q[0]) / (q[4] - q[2]) - 2.91)
    rng = np.random.default_rng(seed)
    rows = []
    for a in ASSETS:
        v = rets[a].values
        sk, ku = qstats(v)
        b = np.array([qstats(v[moving_block_indices(len(v), block, rng)]) for _ in range(n_boot)])
        rows.append({'asset': LABELS[a], 'skew_moment': stats.skew(v), 'skew_hinkley': sk,
                     'skew_lo': np.quantile(b[:, 0], 0.025), 'skew_hi': np.quantile(b[:, 0], 0.975),
                     'exkurt_moment': stats.kurtosis(v), 'kurt_cs': ku,
                     'kurt_lo': np.quantile(b[:, 1], 0.025), 'kurt_hi': np.quantile(b[:, 1], 0.975)})
    t = pd.DataFrame(rows).set_index('asset')
    t.to_csv(os.path.join(TABLE_DIR, 'ch1_robust_moments.csv'), float_format='%.4g')
    return t


def acf_robust_lag1():
    """rho_1 cu banda i.i.d. 1.96/sqrt(T) si banda robusta 1.96 sqrt(tau_1) (Romano & Thombs 1996)."""
    rows = []
    for a in ASSETS:
        x = (rets[a] - rets[a].mean()).values
        T, s2 = len(x), np.sum(x ** 2)
        rho = np.sum(x[1:] * x[:-1]) / s2
        tau = np.sum(x[1:] ** 2 * x[:-1] ** 2) / s2 ** 2
        rows.append({'asset': LABELS[a], 'T': T, 'rho1': rho, 'T_tau1': T * tau, 'band_iid': 1.96 / np.sqrt(T),
                     'band_robust': 1.96 * np.sqrt(tau), 't_iid': rho * np.sqrt(T), 't_robust': rho / np.sqrt(tau)})
    return pd.DataFrame(rows).set_index('asset')


def long_memory_table(power=0.65):
    """d pentru |r_t|: local Whittle si GPH cu m = T^0.65; apoi re-estimare de o parte si de alta
    a rupturii de varianta estimate (verificarea 'memorie lunga sau rupturi?')."""
    rows = []
    for a in ASSETS:
        r = rets[a]
        x = np.abs(r.values)
        T = len(x)
        m = int(T ** power)
        dlw, slw = local_whittle(x, m)
        dg, sg = gph(x, m)
        j, it = cusum_squares_break(r.values)
        pre, post = x[:j + 1], x[j + 1:]
        dpre, spre = local_whittle(pre, int(len(pre) ** power))
        dpost, spost = local_whittle(post, int(len(post) ** power))
        P = PERIODS[a]
        rows.append({'asset': LABELS[a], 'T': T, 'm': m, 'd_lw': dlw, 'se_lw': slw, 'd_gph': dg, 'se_gph': sg,
                     'break_date': r.index[j].date(), 'IT_stat': it,
                     'vol_pre_pct': 100 * r.values[:j + 1].std() * np.sqrt(P),
                     'vol_post_pct': 100 * r.values[j + 1:].std() * np.sqrt(P),
                     'd_pre': dpre, 'se_pre': spre, 'd_post': dpost, 'se_post': spost})
    t = pd.DataFrame(rows).set_index('asset')
    t.to_csv(os.path.join(TABLE_DIR, 'ch1_long_memory.csv'), float_format='%.4g')
    return t

## Run

In [ ]:
TABLE_DIR = '.'
print(robust_moments().round(3).T)
print(acf_robust_lag1().round(3).T)
print(hill_inference().round(3).T)
print(long_memory_table().round(3).T)